# Tutorial on `Setting`s

## 1. Convex Tikhonov `Setting`s

Set up a foward operator -- here a convolution operator, define an exact solution (phantom) and compute corresponding exact data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from regpy.operators import GaussianBlur
from regpy.vecsps import UniformGridFcts


grid = UniformGridFcts((0,1,100),periodic=True)
op = GaussianBlur(grid,sigma=0.1)           # forward operator
x = grid.coords[0]                          # coordinates of sample points
exact_sol = 0.5*(np.abs(x-0.5)<=0.2)         # exact solution
exact_data = op(exact_sol)                  # exact data


Create a setting by specifying penalty and data-fidelity terms. Here we choose and $L^2$ data penalty term with non-negativity constraint and an $L^2$ data fidelity term, i.e., the Tikhonov functional
$$
\frac{1}{2}\|Tf-data\|_{L^2}^2 + \frac{10^{-2}}{2}\|f\|_{L^2}^2 + \chi_{\geq 0}
$$  
where $\chi_{\geq 0}(f):=0$ if $f\geq 0$ and $\chi_{\geq 0}(f):=\infty$ else.

In [ ]:
from regpy.solvers import Setting
from regpy.hilbert import L2
from regpy.functionals import QuadLow

setting = Setting(op, QuadLow, L2,
                  data=exact_data, regpar = 1e-2)

### What happens under the hood

The arguments `QuadLow` and `L2` are *abstract factories*, not yet
functionals on a particular vector space. During initialization, `Setting`
converts them into concrete functionals on the domain and codomain of `op`:

- `QuadLow` is called on `op.domain` (here `grid`) and produces the
  integral functional `QuadraticLowerBound`. It represents
  $\mathcal R(f)=\tfrac12\|f\|_{L^2}^2+\chi_{\geq 0}(f)$, and its
  associated Hilbert space is $L^2$ on `grid`.
- `L2` is called on `op.codomain` to obtain an `L2` **Hilbert space**.
  Because `Setting` expects a functional, it wraps this Hilbert space in
  `SquaredNorm`, giving $\mathcal S(y)=\tfrac12\|y\|_{L^2}^2$.
- The keyword `data=exact_data` shifts the **data-fidelity functional** to
  $\mathcal S_{g}(y)=\tfrac12\|y-g\|_{L^2}^2$, with
  $g=\mathtt{exact\_data}$. In general, data need to belong to the codomain of the operator, 
  and other data fidelty terms incorporate them in other ways than as shifts, e.g., the 
  Kullback-Leibler divergence.  
  The parameter `regpar=1e-2` weights the penalty in the minimization problem.

Thus `Setting(op, QuadLow, L2, ...)` is equivalent to explicitly
constructing these two functionals. The explicit construction below is useful
when you want to inspect or customize their parameters.

In [ ]:
from regpy.functionals.integral_functionals import QuadraticLowerBound
from regpy.functionals.base import SquaredNorm

explicit_penalty = QuadraticLowerBound(op.domain)
explicit_fidelity = SquaredNorm(L2(op.codomain), data=exact_data)
explicit_setting = Setting(
    op, explicit_penalty, explicit_fidelity, regpar=1e-2
)

**noise:** Actually, we usually want to test methods with noisy data. Of course, we can add noise to the exact data manually before passing it to the setting, but it is more convenient to let the setting do the job. Then parameters of the noise model do not need to be passed to stopping rule manually. Of course, the exact data are not used in solvers. 

**initial guess:** You can also specify an initial guess for the solution to the inverse problem. An optional argument incorporate_init_in_penalty defaulting to True specifies if the data should be incorporated into the penalty functional or be only used as starting point of iterative methods. In the first case the penalty must be a data functional, and the incorporation is not necessarily done by a shift, e.g. in RelativeEntropy.  

In [ ]:
setting = Setting(op, QuadLow, L2,
                  exact_data=exact_data, regpar = 1e-2,
                  init = 0.2*grid.ones(), incorporate_init_in_penalty=False)
setting.add_Gaussian_noise(relative_noise_level=0.05, white_noise=False,seed = 197)

`display_all_methods()` informs us about the methods that are available to minimize the Tikhonov functional:

In [ ]:
setting.display_all_methods(only_applicable=True)

Let us choose the semismooth Newton method and plot reconstructions.

In [ ]:
from regpy.solvers.plotting import convergence_plots
reco, reco_data = setting.run('SSN',max_it=100)

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,3))
ax1.plot(x,exact_sol, label = 'exact solution')
ax1.plot(x,reco, label = 'reconstruction')
ax1.legend()
ax2.plot(x,setting.data, label = 'noisy data')
ax2.plot(x,exact_data, label = 'exact data')
ax2.legend()


If we want to plot intermediate iterates, study properties of the solver, etc., we have to instantiate the `Solver` and the `StopRule` manually. Here we track the active set. 

In [ ]:
from regpy.solvers.linear import SemiSmoothNewton
from regpy.stoprules import DualityGapStopping

solver = SemiSmoothNewton(setting)
stoprule = DualityGapStopping(schedule=None) 
# By default, DualityGapStopping gets a schedule to check at every 10th step 
# as the evaluation of this stopping criterion may require expensive operator evaluations. 
# Here we do not want a schedule, which is the default for SemiSmoothNewton. 

# reco, reco_data = solver.run(stoprule, max_it=100)
# This would be identical to letting setting run the solver. 

plt.figure(figsize=(10, 1.5))
for reco, reco_data in solver.while_(stoprule, max_it=100):
    active = solver.active_minus
    plt.plot(x[active],solver.iteration_step_nr*np.ones(np.sum(active)),'.')

## 2. `Setting`s without a regularization parameter

If we delete the regularization parameter, solvers try to compute an approximate solution to the inverse problem based on qualitative information on the solution provided by the penalty functional. (This is not necessarily the case for settings with a regularization parameter unless this parameter is chosen properly!) Hence, the set of solvers is disjoint. 

In [ ]:
del setting.regpar
# This yields the same result as omitting the argument `regpar` in the constructor. 
setting.display_all_methods(only_applicable=True)

We can apply Tikhonov regularization on a grid of regularization parameters and use a parameter choice rule (by default the `Discrepancy` principle) to select a suitable regularization parameters. Alternatively, we can solve the problem by an iterative method using the Hilbert geometry specified by the data and penalty term and the constraints. Here we apply FISTA with penalty $\chi_{\geq 0}$. 

To configure `Solver`s and their `StopRule`s, use `set_method_params`, `get_method_params`, `set_stoprule_params`, and `set_stoprule_params`. Parameters are always passed as dictionaries.

`StopRule`s are also used for monitoring and may contain history dictionary after the solver they are attached to has been run. Use `get_stoprule` retrieve a StopRule. For standard convergence plots you may also use RegPy's `convergence_plots` routine.  

In [ ]:
from regpy.solvers import Naturals
from regpy.solvers.plotting import convergence_plots

methods = ['alGr', 'nFISTA']

setting.set_method_params('alGr',{'regpars':0.1*0.5**Naturals(),
                                  'logging_level':'WARNING'})
setting.set_stoprule_params(methods,{'tau':1.2,
                                     'logging_level':'WARNING'})

reco = {method: setting.run(method, ref_sol=(exact_sol,exact_data), max_it=100)[0] 
        for method in methods
        }

fig, (ax1,ax2) = plt.subplots(1,2,figsize=[15,4])
for method in methods:
    ax1.plot(x,reco[method],label=method)
ax1.plot(x,exact_sol,label='exact')
ax1.legend()
_= convergence_plots(setting, x_axis = 'op_evals',axes = ax2)


## 3. `Setting`s with nonlinear operators

Let us define a nonlinear operator between the same spaces: 
$$
F(f) := |\exp(FAC*\Delta) \exp(if)|^2
$$
where $\exp(FAC*\Delta)$ denotes a functional calculus of the Laplacian and $FAC=10^{-4}+10^{-2}i$. 

In [ ]:
from regpy.operators import Laplacian, NemytskiiOperator, SquaredModulus

FAC =0.01j+0.0001

cmplx_grid = grid.complex_space()
Lap = Laplacian(cmplx_grid)
conv = Lap.functional_calculus(lambda x: np.exp(FAC*x))

f=lambda x:np.exp(1j*x)
Df =  lambda x:np.stack((-np.sin(x), np.cos(x)), axis=-1)[..., None]

F = SquaredModulus(cmplx_grid) * conv * NemytskiiOperator(grid,f=f, Df=Df)
exact_nl_data = F(exact_sol)

nl_sett = Setting(F, QuadLow, L2, exact_data = exact_nl_data)
# here we assume the data to be Poisson distributed
nl_sett.generate_Poisson_data(expected_nr_counts=1e6)

nl_sett.display_all_methods(only_applicable=True)

In [ ]:
nl_methods = ['gFISTA','IRN']

# configure methods
nl_sett._default_inner_solver = 'SSN' # choose inner solver for IRN
nl_sett.set_method_params(nl_methods,{'logging_level':'WARNING'})
nl_sett.set_stoprule_params(nl_methods,{'tau':1.2,
                                        'logging_level':'WARNING'})

for method in nl_methods:
    nl_reco = {method: nl_sett.run(method, ref_sol=(exact_sol,exact_nl_data), max_it = 30)[0] 
            for method in nl_methods}

fig, (ax1,ax2) = plt.subplots(1,2,figsize=[15,4])
for method in nl_methods:
    ax1.plot(x,nl_reco[method],label=method)
ax1.plot(x,exact_sol,label='exact')
ax1.legend()
_= convergence_plots(nl_sett, x_axis = 'op_evals',axes = ax2)


## 4. Linearized and dual `Setting`s

The methods `IRN` and `nlBreg` repeatedly generate linearized Tikhonov settings in which convex inner solvers are run. A linearized setting at a point $f_0$ in the operator's domain has the following form.

Let $F:X\to Y$ be the nonlinear forward operator, let $g^\delta$ be the measured data, and denote its derivative at $f_0$ by $A=F'[f_0]$. Instead of solving for the entire reconstruction $f$, an inner solver computes the **correction** $h=f-f_0$. Linearizing the forward operator,

$$
F(f_0+h)\approx F(f_0)+Ah,
$$

gives the convex Tikhonov subproblem (when the data fidelity $\mathcal S_{g^\delta}$ and penalty $\mathcal R$ are convex)

$$
\boxed{\min_h\;\mathcal S_{g^\delta}\bigl(F(f_0)+Ah\bigr)
      +\alpha\,\mathcal R(f_0+h).}
$$

For the quadratic $L^2$ data fidelity used here, this becomes

$$
\min_h\;\frac12\bigl\|Ah-(g^\delta-F(f_0))\bigr\|_{L^2}^2
      +\alpha\,\mathcal R(f_0+h).
$$

`get_linearized_tikh_setting(x0=f0, regpar=alpha)` constructs precisely this problem: its operator is $A$, its data-fidelity functional incorporates the shift by $F(f_0)$, and its penalty is expressed in **correction coordinates**, i.e., $h\mapsto\mathcal R(f_0+h)$. The default initial correction is $h=0$. Therefore, the result returned by an inner solver is a correction $h$; the corresponding reconstruction is $f_0+h$. In the example below $f_0=0$, so the correction and reconstruction coincide.

The method also accepts a previously computed value and derivative (`y0`, `deriv`) to avoid recomputation. With `use_bregman_penalty=True`, the penalty is replaced by the corresponding Bregman distance. It may be advisable to test inner solvers on such a linearized setting in order to choose and configure one. You can use `set_method_params` and `set_stoprule_params` for inner solvers as well.

In [ ]:
lin_sett = nl_sett.get_linearized_tikh_setting(x0 = grid.zeros(),regpar = 1e-2)
lin_sett.display_all_methods(only_applicable=True)

In [ ]:
lin_methods = ['FISTA','dFISTA','PDHG','dPDHG','SSN']

lin_sett.set_method_params(lin_methods,{'logging_level':'WARNING'})
lin_sett.set_stoprule_params(lin_methods,{'logging_level':'WARNING'})

lin_rec = {method: lin_sett.run(method,max_it=1000)[0] for method in lin_methods}

_= convergence_plots(lin_sett,x_axis = 'op_evals')

### The dual setting

For a **linear** forward operator $T$ and a **convex** Tikhonov setting with a positive regularization parameter $\alpha$, RegPy also provides the Fenchel dual problem. Writing the primal minimization problem as

$$
\min_f\;\frac{1}{\alpha}\mathcal S_{g^\delta}(Tf)+\mathcal R(f),
$$

the dual, in RegPy's sign convention, can be written as the minimization problem

$$
\boxed{\min_p\;\mathcal R^*(T^*p)
      +\frac{1}{\alpha}\mathcal S_{g^\delta}^*(-\alpha p).}
$$

Here $T^*$ is the adjoint operator, and $\mathcal R^*$ and $\mathcal S_{g^\delta}^*$ are the Fenchel conjugates of the penalty and data fidelity. RegPy builds the dual setting automatically from these components:

```python
# lin_sett is linear, convex, and has a positive regularization parameter.
dual_sett = lin_sett.get_dual_setting()
dual_sett.display_all_methods()
```

One can therefore inspect or configure the dual minimization problem independently. The primal setting also provides `primal_to_dual` and `dual_to_primal` for converting compatible primal and dual variables when the required subgradients are available.

**Normally, however, there is no need to create `dual_sett` explicitly.** When an appropriate method is available, calling `lin_sett.run('dFISTA')` or `lin_sett.run('dPDHG')`, as in the preceding cell, selects the corresponding dual algorithm and handles the dual setting internally.